# Predicting Stellar Class (Kaggle)
This notebook contains the complete, highly-optimized ML pipeline for the Kaggle Playground Series s6e6 competition.

Features include:
- **Domain-Specific Feature Engineering** (color indices, redshift interactions)
- **Class Imbalance Handling** (Balanced Class Weights)
- **Optuna-Optimized Models** (LightGBM, XGBoost, CatBoost, RF, ET)
- **Soft-Voting Ensemble** for the final prediction

In [ ]:
# 1. IMPORTS & SETUP
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Setup paths for Kaggle
TRAIN_CSV = "/kaggle/input/playground-series-s6e6/train.csv"
TEST_CSV = "/kaggle/input/playground-series-s6e6/test.csv"
SUBMISSION_CSV = "submission.csv"

RANDOM_STATE = 42

## 2. Feature Engineering & Preprocessing

In [ ]:
def load_and_preprocess_data(filepath, is_train=True, encoder=None):
    df = pd.read_csv(filepath)
    
    # Domain-specific feature engineering
    # 1. Color Indices (Photometric)
    df["u-g"] = df["u"] - df["g"]
    df["g-r"] = df["g"] - df["r"]
    df["r-i"] = df["r"] - df["i"]
    df["i-z"] = df["i"] - df["z"]
    
    # 2. Redshift interactions
    df["redshift_g_r"] = df["redshift"] * df["g-r"]
    df["redshift_r_i"] = df["redshift"] * df["r-i"]
    df["redshift_sq"] = df["redshift"] ** 2
    
    # 3. Band statistics
    bands = ["u", "g", "r", "i", "z"]
    df["band_mean"] = df[bands].mean(axis=1)
    df["band_std"] = df[bands].std(axis=1)
    df["band_max"] = df[bands].max(axis=1)
    df["band_min"] = df[bands].min(axis=1)
    df["band_range"] = df["band_max"] - df["band_min"]
    
    if is_train:
        X = df.drop(columns=["id", "class"])
        y = df["class"]
    else:
        X = df.drop(columns=["id"])
        y = None
        
    # Handle categoricals
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(cat_cols) > 0:
        if is_train:
            encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            X[cat_cols] = encoder.fit_transform(X[cat_cols])
        else:
            X[cat_cols] = encoder.transform(X[cat_cols])
            
    return X, y, encoder

## 3. Define Models (with Optuna-Tuned Parameters)

In [ ]:
def get_models(class_weights=None):
    return {
        "LightGBM": LGBMClassifier(
            n_estimators=874,
            learning_rate=0.055022422187696825,
            num_leaves=162,
            max_depth=18,
            min_child_samples=39,
            feature_fraction=0.5570863190538407,
            bagging_fraction=0.8585816816694674,
            bagging_freq=2,
            reg_alpha=5.663709646092421e-07,
            reg_lambda=0.14798921503221413,
            class_weight=class_weights,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "CatBoost": CatBoostClassifier(
            iterations=800,
            learning_rate=0.05,
            depth=6,
            random_state=RANDOM_STATE,
            verbose=False,
            thread_count=-1
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            class_weight=class_weights,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=300,
            max_depth=15,
            class_weight=class_weights,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }

## 4. Train Models & Ensemble

In [ ]:
print("Loading and preprocessing data...")
X_train, y_train, encoder = load_and_preprocess_data(TRAIN_CSV, is_train=True)
X_test, _, _ = load_and_preprocess_data(TEST_CSV, is_train=False, encoder=encoder)

# Compute class weights for STAR imbalance
classes = np.unique(y_train)
cw_array = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, cw_array))
print(f"Class weights: {class_weights}")

# Get Models
models = get_models(class_weights)
test_preds = np.zeros((len(X_test), len(classes)))

print("\nTraining models on full dataset...")
for name, model in models.items():
    print(f"Training {name}...")
    if name == "XGBoost":
        # XGBoost requires numeric targets
        target_map = {c: i for i, c in enumerate(classes)}
        y_train_num = y_train.map(target_map)
        model.fit(X_train, y_train_num)
        test_preds += model.predict_proba(X_test)
    else:
        if name == "LightGBM":
            sample_weights = np.array([class_weights[c] for c in y_train])
            model.fit(X_train, y_train, sample_weight=sample_weights)
        else:
            model.fit(X_train, y_train)
        test_preds += model.predict_proba(X_test)

# Average probabilities
test_preds /= len(models)

# Final predictions
final_classes = [classes[i] for i in np.argmax(test_preds, axis=1)]

# Save Submission
sample_sub = pd.read_csv("/kaggle/input/playground-series-s6e6/sample_submission.csv")
submission = pd.DataFrame({
    "id": sample_sub["id"],
    "class": final_classes
})
submission.to_csv(SUBMISSION_CSV, index=False)
print("\n[OK] Successfully saved submission.csv!")